In [2]:
import os
import numpy as np
import torch
from utils.metrics import *
from collections import OrderedDict
from tqdm import tqdm

def euclidean_distance_matrix(matrix1, matrix2):
    assert matrix1.shape[1] == matrix2.shape[1]
    d1 = -2 * np.dot(matrix1, matrix2.T)
    d2 = np.sum(np.square(matrix1), axis=1, keepdims=True)
    d3 = np.sum(np.square(matrix2), axis=1)
    dists = np.sqrt(d1 + d2 + d3)
    return dists

def sort_by_dist(data_list, reverse=False):
    sortlist = sorted(data_list, key=lambda x: x['dist'], reverse=reverse)
    for one in sortlist:
        del one['dist']
    return sortlist

def get_top10_similar_mofea_features(audio_feat, eval_wrapper, motionembedding_dir, device='cuda:0'):
    """
    Params:
        audio_feat: numpy.ndarray, shape (384, 264) - 单个音频滑窗特征
        eval_wrapper: EvaluatorModelWrapper 实例 - 用于生成音频嵌入
        motionembedding_dir: str - motion 特征的 numpy 存储目录（如 '/data1/.../motionembedding'）
        device: str - 如 'cuda:0'
    
    Returns:
        sorted_top10: list[dict] - 每个dict包含 { 'name': motion_name, 'embedding': 特征（可选） }
    """
    assert audio_feat.shape == (384, 55), "Input audio feature shape must be (384, 264)"
    audio_feat_tensor = torch.from_numpy(audio_feat).unsqueeze(0).to(device).to(torch.float32)
    
    audio_batch = ("query_sample", audio_feat_tensor, torch.zeros([1, 55], device=device))  # shape (1, 384, 264)
    with torch.no_grad():
        audio_embedding = eval_wrapper.get_co_embeddings(audio_batch, "audio").cpu().numpy()  # shape (1, D)

    # 加载所有 motion embeddings
    all_motion_embeddings = []
    seqnames = []
    for file in os.listdir(motionembedding_dir):
        if not file.endswith('.npy'):
            continue
        motion_emb = np.load(os.path.join(motionembedding_dir, file))
        all_motion_embeddings.append(motion_emb)
        seqnames.append(file.replace('.npy', ''))
    
    all_motion_embeddings = np.stack(all_motion_embeddings, axis=0)  # shape (N, D)
    
    dist_mat = euclidean_distance_matrix(audio_embedding, all_motion_embeddings)  # shape (1, N)
    argsmax = np.argsort(dist_mat[0])[:10]  # top 10 index

    top10 = []
    for idx in argsmax:
        top10.append({
            'name': seqnames[idx],
            'dist': dist_mat[0][idx],
            # 'embedding': all_motion_embeddings[idx]  # 可选
        })

    sorted_top10 = sort_by_dist(top10)
    return sorted_top10
from configs import get_config
from datasets import EvaluatorModelWrapper

motionembedding_dir = '/data1/hzy/AllDataset/motionembeding'
config_path = '/data1/hzy/HumanMotion/RetrievalNet/checkpoints/AInterClip_Audio55Motion264/0512/train/bc256_s100l384_drop0.2_lr1e-4/InterCLIP.yaml'
device = 'cuda:0'

# 初始化模型包装器
eval_wrapper = EvaluatorModelWrapper(get_config(config_path), device)

# 加载你的音频特征 (384, 55)
audio_feat = np.load('/data1/hzy/AllDataset/musicfeature_55_allmusic_pure/_P-JWcq1ewI_01_1381_1800.npy')[0:384].astype(np.float32)

# 获取 top 10 相似 motion
top10_results = get_top10_similar_mofea_features(audio_feat, eval_wrapper, motionembedding_dir, device=device)

# 打印结果
for i, result in enumerate(top10_results):
    print(f"Top {i+1}: {result['name']}")


Top 1: _P-JWcq1ewI_01_1381_1800@0_384
Top 2: hi7N4C5TgIw_01_0_1770@1300_1684
Top 3: C4B7ZDi1P10_05_0_1390@900_1284
Top 4: C4B7ZDi1P10_05_0_1390@1000_1384
Top 5: 9eKgqkPUs1I_28_0_1560@0_384
Top 6: 9eKgqkPUs1I_28_0_1560@100_484
Top 7: WfAHqpvVIrg_16_0_1650@400_784
Top 8: C4B7ZDi1P10_05_0_1390@200_584
Top 9: WfAHqpvVIrg_16_0_1650@300_684
Top 10: C4B7ZDi1P10_05_0_1390@800_1184


In [4]:
import numpy as np

old = np.load("/data1/hzy/AllDataset/musicfeature_55_allmusic_pure_old60/_P-JWcq1ewI_03_0_1350.npy")
new = np.load("/data1/hzy/AllDataset/musicfeature_55_allmusic_pure/_P-JWcq1ewI_03_0_1350.npy")

print("Old shape:", old.shape, "dtype:", old.dtype)
print("New shape:", new.shape, "dtype:", new.dtype)


Old shape: (2701, 55) dtype: float64
New shape: (1351, 55) dtype: float64


In [3]:
import os
import json
import numpy as np
import torch
from collections import Counter
from tqdm import tqdm
from utils.metrics import euclidean_distance_matrix
from configs import get_config
from datasets import EvaluatorModelWrapper

def sort_by_dist(data_list, reverse=False):
    sortlist = sorted(data_list, key=lambda x: x['dist'], reverse=reverse)
    for one in sortlist:
        del one['dist']
    return sortlist

def get_top10_similar_mofea_features(audio_feat, eval_wrapper, motionembedding_dir, device='cuda:0'):
    assert audio_feat.shape == (384, 55), "Input audio feature shape must be (384, 55)"
    audio_feat_tensor = torch.from_numpy(audio_feat).unsqueeze(0).to(device).to(torch.float32)

    audio_batch = ("query_sample", audio_feat_tensor, torch.zeros([1, 55], device=device))
    with torch.no_grad():
        audio_embedding = eval_wrapper.get_co_embeddings(audio_batch, "audio").cpu().numpy()

    all_motion_embeddings = []
    seqnames = []
    for file in os.listdir(motionembedding_dir):
        if file.endswith('.npy'):
            motion_emb = np.load(os.path.join(motionembedding_dir, file))
            all_motion_embeddings.append(motion_emb)
            seqnames.append(file.replace('.npy', ''))

    all_motion_embeddings = np.stack(all_motion_embeddings, axis=0)

    dist_mat = euclidean_distance_matrix(audio_embedding, all_motion_embeddings)
    argsmax = np.argsort(dist_mat[0])[:10]

    top10 = []
    for idx in argsmax:
        top10.append({
            'name': seqnames[idx],
            'dist': dist_mat[0][idx],
        })

    sorted_top10 = sort_by_dist(top10)
    return sorted_top10

def get_style_representation(name: str = None,
                              motion: np.ndarray = None,
                              idx: int = 0,
                              eval_wrapper=None,
                              motionembedding_dir="/data1/hzy/AllDataset/motionembeding",
                              retrieval_path="/data1/hzy/AllDataset/retrieval_s100_l384",
                              motion_base="/data1/hzy/HumanMotion/All_mofea/alldata_new_joint_vecs264",
                              style_map_path="/data1/hzy/HumanMotion/All_mofea/styles/all_style_map.json",
                              meta_path="meta",
                              device='cuda:0'):
    """
    支持两种输入：
    - name + idx：优先读取对应 JSON
    - motion: 若 JSON 不存在，则根据 motion 计算 top10
    """

    if name:
        json_path = os.path.join(retrieval_path, f"{name}.json")
        if os.path.exists(json_path):
            with open(json_path, 'r') as f:
                data = json.load(f)

            musiclist = data[idx][:10]
            top10_results = [{'name': item['name']} for item in musiclist]
        else:
            if motion is None:
                raise ValueError("motion is required when JSON file is not available")
            top10_results = get_top10_similar_mofea_features(motion, eval_wrapper, motionembedding_dir, device=device)
    else:
        if motion is None:
            raise ValueError("Either 'name' or 'motion' must be provided.")
        top10_results = get_top10_similar_mofea_features(motion, eval_wrapper, motionembedding_dir, device=device)

    # 提取 motion 片段
    results = []
    for item in top10_results:
        full_name = item['name']
        if '@' in full_name:
            video_part, frame_part = full_name.rsplit('@', 1)
            if '_' in frame_part:
                start_frame, end_frame = map(int, frame_part.split('_'))
                results.append((video_part, start_frame, end_frame))

    # 加载 style map
    with open(style_map_path, 'r') as f:
        style_map = json.load(f)

    genres = []
    for video_name, _, _ in results:
        genre = style_map.get(video_name, 'Unknown')
        genres.append(genre)

    genre_counts = Counter(genres)
    total = len(genres)
    genre_proportions = {genre: count / total for genre, count in genre_counts.items()}
    top_genre = max(genre_proportions, key=genre_proportions.get, default='Unknown')

    # Load Mean & Std
    mean = np.load(os.path.join(motion_base, meta_path, "Mean.npy"))
    std = np.load(os.path.join(motion_base, meta_path, "Std.npy"))

    mofea264_list = []
    for video_name, start, end in results:
        motion_path = os.path.join(motion_base, f"{video_name}.npy")
        if not os.path.exists(motion_path):
            print(f"Warning: {motion_path} not found, skipping...")
            continue

        motion_data = np.load(motion_path)
        if end > motion_data.shape[0]:
            print(f"Warning: {video_name} range ({start}-{end}) exceeds motion length {motion_data.shape[0]}")
            continue

        segment = motion_data[start:end]
        normed_segment = (segment - mean) / std
        mofea264_list.append(normed_segment)

    if not mofea264_list:
        print("No valid motion segments found.")
        return None, genre_proportions, top_genre, top10_results

    weights = np.array([10 - i for i in range(min(len(mofea264_list), 10))], dtype=np.float32)
    weights /= weights.sum()

    min_length = min(segment.shape[0] for segment in mofea264_list)
    aligned_segments = [segment[:min_length] for segment in mofea264_list]

    weighted_sum = np.zeros_like(aligned_segments[0], dtype=np.float32)
    for i, segment in enumerate(aligned_segments):
        weighted_sum += weights[i] * segment
    top10_names_indices = []
    for item in top10_results:
        full_name = item['name']
        if '@' in full_name:
            video_part, frame_part = full_name.rsplit('@', 1)
            if '_' in frame_part:
                start_idx, end_idx = map(int, frame_part.split('_'))
                top10_names_indices.append((video_part, start_idx, end_idx))
                
    # top10_names_indices
    motiontoken_dir = "/data1/hzy/HumanMotion/All_mofea/Alldata/MotionTokens_512_vel_processed"
    token_segments = []

    for name, start_idx, _ in top10_names_indices:
        token_file = os.path.join(motiontoken_dir, f"{name}.npy")
        if not os.path.exists(token_file):
            print(f"Warning: Token file {token_file} not found.")
            continue

        tokens = np.load(token_file)  # shape: (N,)

        token_index = round(start_idx / 96) * 72
        if token_index + 60 > len(tokens):
            print(f"Warning: Token index {token_index} out of range for file {name}.npy (len={len(tokens)}).")
            continue

        token_segment = tokens[token_index:token_index + 60]
        token_segments.append({
            "name": name,
            "start_idx": start_idx,
            "token_index": token_index,
            "tokens": token_segment.tolist(),  # or keep as np.array if needed
        })

    # 输出结果
    print("\nExtracted token segments (first 60 tokens each):")
    for i, item in enumerate(token_segments):
        print(f"Top {i+1}: {item['name']} @ token_index={item['token_index']}, tokens[:5]={item['tokens'][:5]}")


    return weighted_sum, genre_proportions, top_genre, top10_results,top10_names_indices,token_segments
# 方法1：通过 JSON 路径读取

#/data1/hzy/AllDataset/retrieval_s100_l384/{name}.json和idx都不为空
music_name="_P-JWcq1ewI_01_1381_18001"
music_idx=0
music_feat = np.load("/data1/hzy/AllDataset/musicfeature_55_allmusic_pure/_P-JWcq1ewI_01_1381_1800.npy")[0:384]
json_path = f"/data1/hzy/AllDataset/retrieval_s100_l384/{music_name}.json"


if os.path.exists(json_path):
    weighted_sum, genre_proportions, top_genre, top10_results,top10_names_indices,token_segments = get_style_representation(name=music_name, idx=music_idx)
else:
    # 如果 JSON 不存在，则使用 motion 特征计算相似性
    
    weighted_sum, genre_proportions, top_genre, top10_results,top10_names_indices,token_segments = get_style_representation(
        motion=music_feat,
        eval_wrapper=eval_wrapper,
        motionembedding_dir="/data1/hzy/AllDataset/motionembeding"
    )
print(weighted_sum.shape)
print(genre_proportions)
for i, result in enumerate(top10_results):
    print(f"Top {i+1}: {result['name']}")
    
print("Top10 Name/Index tuples:")
for i, (n, s, e) in enumerate(top10_names_indices):
    print(f"Top {i+1}: {n}, {s}, {e}")
print("Token Segments:",token_segments[0]["tokens"])
    


NameError: name 'eval_wrapper' is not defined

In [1]:
import os
import json
import numpy as np
import torch
from collections import Counter
from tqdm import tqdm
from utils.metrics import euclidean_distance_matrix
from configs import get_config
from datasets import EvaluatorModelWrapper

def sort_by_dist(data_list, reverse=False):
    sortlist = sorted(data_list, key=lambda x: x['dist'], reverse=reverse)
    for one in sortlist:
        del one['dist']
    return sortlist

def get_top10_similar_mofea_features(audio_feat, eval_wrapper, motionembedding_dir, device='cuda:0'):
    assert audio_feat.shape == (384, 55), "Input audio feature shape must be (384, 55)"
    audio_feat_tensor = torch.from_numpy(audio_feat).unsqueeze(0).to(device).to(torch.float32)

    audio_batch = ("query_sample", audio_feat_tensor, torch.zeros([1, 55], device=device))
    with torch.no_grad():
        audio_embedding = eval_wrapper.get_co_embeddings(audio_batch, "audio").cpu().numpy()

    all_motion_embeddings = []
    seqnames = []
    for file in os.listdir(motionembedding_dir):
        if file.endswith('.npy'):
            motion_emb = np.load(os.path.join(motionembedding_dir, file))
            all_motion_embeddings.append(motion_emb)
            seqnames.append(file.replace('.npy', ''))

    all_motion_embeddings = np.stack(all_motion_embeddings, axis=0)

    dist_mat = euclidean_distance_matrix(audio_embedding, all_motion_embeddings)
    argsmax = np.argsort(dist_mat[0])[:10]

    top10 = []
    for idx in argsmax:
        top10.append({
            'name': seqnames[idx],
            'dist': dist_mat[0][idx],
        })

    sorted_top10 = sort_by_dist(top10)
    return sorted_top10

def get_style_representation(name: str = None,
                              motion: np.ndarray = None,
                              idx: int = 0,
                              eval_wrapper=None,
                              motionembedding_dir="/data1/hzy/AllDataset/motionembeding",
                              retrieval_path="/data1/hzy/AllDataset/retrieval_s100_l384",
                              motion_base="/data1/hzy/HumanMotion/All_mofea/alldata_new_joint_vecs264",
                              style_map_path="/data1/hzy/HumanMotion/All_mofea/styles/all_style_map.json",
                              meta_path="meta",
                              device='cuda:0'):
    """
    支持两种输入：
    - name + idx：优先读取对应 JSON
    - motion: 若 JSON 不存在，则根据 motion 计算 top10
    """

    if name:
        json_path = os.path.join(retrieval_path, f"{name}.json")
        if os.path.exists(json_path):
            with open(json_path, 'r') as f:
                data = json.load(f)

            musiclist = data[idx][:10]
            top10_results = [{'name': item['name']} for item in musiclist]
        else:
            if motion is None:
                raise ValueError("motion is required when JSON file is not available")
            top10_results = get_top10_similar_mofea_features(motion, eval_wrapper, motionembedding_dir, device=device)
    else:
        if motion is None:
            raise ValueError("Either 'name' or 'motion' must be provided.")
        top10_results = get_top10_similar_mofea_features(motion, eval_wrapper, motionembedding_dir, device=device)

    # 提取 motion 片段
    results = []
    for item in top10_results:
        full_name = item['name']
        if '@' in full_name:
            video_part, frame_part = full_name.rsplit('@', 1)
            if '_' in frame_part:
                start_frame, end_frame = map(int, frame_part.split('_'))
                results.append((video_part, start_frame, end_frame))

    # 加载 style map
    with open(style_map_path, 'r') as f:
        style_map = json.load(f)

    genres = []
    for video_name, _, _ in results:
        genre = style_map.get(video_name, 'Unknown')
        genres.append(genre)

    genre_counts = Counter(genres)
    total = len(genres)
    genre_proportions = {genre: count / total for genre, count in genre_counts.items()}
    top_genre = max(genre_proportions, key=genre_proportions.get, default='Unknown')

    # Load Mean & Std
    mean = np.load(os.path.join(motion_base, meta_path, "Mean.npy"))
    std = np.load(os.path.join(motion_base, meta_path, "Std.npy"))

    mofea264_list = []
    for video_name, start, end in results:
        motion_path = os.path.join(motion_base, f"{video_name}.npy")
        if not os.path.exists(motion_path):
            print(f"Warning: {motion_path} not found, skipping...")
            continue

        motion_data = np.load(motion_path)
        if end > motion_data.shape[0]:
            print(f"Warning: {video_name} range ({start}-{end}) exceeds motion length {motion_data.shape[0]}")
            continue

        segment = motion_data[start:end]
        normed_segment = (segment - mean) / std
        mofea264_list.append(normed_segment)

    if not mofea264_list:
        print("No valid motion segments found.")
        return None, genre_proportions, top_genre, top10_results

    weights = np.array([10 - i for i in range(min(len(mofea264_list), 10))], dtype=np.float32)
    weights /= weights.sum()

    min_length = min(segment.shape[0] for segment in mofea264_list)
    aligned_segments = [segment[:min_length] for segment in mofea264_list]

    weighted_sum = np.zeros_like(aligned_segments[0], dtype=np.float32)
    for i, segment in enumerate(aligned_segments):
        weighted_sum += weights[i] * segment

    return weighted_sum, genre_proportions, top_genre, top10_results
weighted_sum, genre_proportions, top_genre, top10_results=

SyntaxError: invalid syntax (<ipython-input-1-7349331fd866>, line 140)